# Tablet detector (owner: P4)
Fine-tune YOLO11n so it detects tablets (COCO has no tablet class).

## 1. Data
Tablet dataset from Roboflow Universe + a few of our own photos. Export in YOLO format.

**Dataset:** [`tablet_dataset`](https://universe.roboflow.com/ihsan-q63h9/tablet_dataset) by `ihsan-q63h9` on Roboflow Universe, **Public Domain**.

- 649 images: 454 train / 130 valid / 65 test
- One class, renamed from `tablet_computer` to `tablet` (the `sed` line below)

The Roboflow API key is read from **Colab Secrets** (`ROBOFLOW_API_KEY`). It is never written in this notebook.

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("ihsan-q63h9").project("tablet_dataset")
dataset = project.version(1).download("yolov11")

In [ ]:
!sed -i "s/tablet_computer/tablet/" {dataset.location}/data.yaml
!cat {dataset.location}/data.yaml

## 2. Train
`YOLO('yolo11n.pt').train(data=..., epochs=50, imgsz=640)`

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
model.train(data=f"{dataset.location}/data.yaml", epochs=50, imgsz=640)

## 3. Evaluate
mAP50 and precision/recall on the test split.

In [ ]:
best = YOLO("/content/runs/detect/train/weights/best.pt")
print(best.names)

metrics = best.val(data=f"{dataset.location}/data.yaml", split="test")
print(metrics.box.map50, metrics.box.p[0], metrics.box.r[0])

**Results (test split: 65 images, 77 tablets)**

| Metric | Value |
|---|---|
| Precision | 0.973 |
| Recall | 0.987 |
| mAP50 | 0.983 |
| mAP50-95 | 0.874 |

> **Note:** these numbers come from a public stock-photo dataset, not our room camera. Next we will add 30–50 photos from our own room and retrain.

## 4. Export
Export to OpenVINO and copy the weights to `backend/weights/`.

In [ ]:
best.export(format="openvino")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/runs/detect/train/weights/best.pt /content/drive/MyDrive/tablet_best.pt
!zip -r /content/drive/MyDrive/tablet_openvino.zip /content/runs/detect/train/weights/best_openvino_model

**Where the weights go:** `perception.py` looks in `backend/weights/` for the first of
`tablet_openvino_model`, `tablet.pt`, `yolo11n_tablet.pt`, `best.pt`.
Rename `tablet_best.pt` to `backend/weights/tablet.pt`.
`perception.py` uses `IMGSZ = 480`, so an OpenVINO export should use `imgsz=480`.
Weights are never committed (`.gitignore`).